In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.3/273.3 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212

In [3]:
!pip install kaggle

In [6]:
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/cs231n/assignments/4/kaggle.json ~/.kaggle/kaggle.json
! chmod 600 ~/.kaggle/kaggle.json

In [7]:
!kaggle competitions download -c walmart-recruiting-store-sales-forecasting
!unzip -q walmart-recruiting-store-sales-forecasting.zip

100% 2.70M/2.70M [00:00<00:00, 208MB/s]



In [8]:
!unzip -q train.csv.zip
!unzip -q stores.csv.zip
!unzip -q test.csv.zip
!unzip -q features.csv.zip

unzip:  cannot find or open stores.csv.zip, stores.csv.zip.zip or stores.csv.zip.ZIP.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from xgboost import XGBRegressor

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
stores = pd.read_csv('stores.csv')
features = pd.read_csv('features.csv')

In [13]:
import dagshub
import mlflow

dagshub.init(repo_owner='tsarc21', repo_name='Walmart-Recruiting---Store-Sales-Forecasting', mlflow=True)


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=0d3cbd22-ec5f-49a0-9fc5-3d798868a2a3&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=8cc1e8f2f8e40d35aa150755cbe24b0f68bc1fbc2d4f93fad900afb451051680




Accessing as tsarc21

Initialized MLflow to track repo "tsarc21/Walmart-Recruiting---Store-Sales-Forecasting"

Repository tsarc21/Walmart-Recruiting---Store-Sales-Forecasting initialized!

In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
merged_data = merged_data.sort_values(
    ["Store", "Dept", "Date"]
)
merged_data["Lag_52"] = (
    merged_data
    .groupby(
        ["Store", "Dept"]
    )["Weekly_Sales"]
    .shift(52)
)
merged_data["Holiday_Lag52"] = (
    merged_data["IsHoliday"].astype(int) * merged_data["Lag_52"]
)
merged_data["Year"] = merged_data["Date"].dt.year
merged_data["Month"] = merged_data["Date"].dt.month
merged_data["Week"] = merged_data["Date"].dt.isocalendar().week.astype(int)
merged_data["Quarter"] = merged_data["Date"].dt.quarter
merged_data["Days_from_start"] = (
    merged_data["Date"] - merged_data["Date"].min()
).dt.days



print(
    "Available Lag values:",
    merged_data["Lag_52"].notna().sum()
)

Available Lag values: 261083


In [ ]:
split_date = "2011-08-01"


train_df = merged_data[
    merged_data["Date"] < split_date
].copy()


val_df = merged_data[
    merged_data["Date"] >= split_date
].copy()


print("Train:", train_df.shape)
print("Validation:", val_df.shape)

Train: (228838, 23)
Validation: (192732, 23)


In [ ]:
train_df = train_df.dropna(
    subset=["Lag_52"]
)


val_df = val_df.dropna(
    subset=["Lag_52"]
)


print("Train after lag:", train_df.shape)
print("Validation after lag:", val_df.shape)

Train after lag: (73092, 23)
Validation after lag: (187991, 23)


In [ ]:
y_train = train_df.pop(
    "Weekly_Sales"
)


y_val = val_df.pop(
    "Weekly_Sales"
)


train_df = train_df.drop(
    columns=["Date"]
)

val_df = val_df.drop(
    columns=["Date"]
)


print(train_df.shape)
print(val_df.shape)

(73092, 21)
(187991, 21)


In [ ]:
class MarkdownFiller(
    BaseEstimator,
    TransformerMixin
):

    def __init__(self):
        self.markdown_cols = [
            "MarkDown1",
            "MarkDown2",
            "MarkDown3",
            "MarkDown4",
            "MarkDown5"
        ]


    def fit(self, X, y=None):
        return self


    def transform(self, X):

        X = X.copy()

        for col in self.markdown_cols:
            if col in X.columns:
                X[col] = X[col].fillna(0)

        return X

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[

        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            make_column_selector(
                dtype_include=[
                    "object",
                    "category",
                    "bool"
                ]
            )
        ),


        (
            "numerical",
            SimpleImputer(
                strategy="median"
            ),
            make_column_selector(
                dtype_include=np.number
            )
        )
    ]
)

In [ ]:
xgb_model = XGBRegressor(
    objective="reg:squarederror",

    random_state=42,

    tree_method="hist",
    device="cuda"
)

In [ ]:
pipeline = Pipeline(
    steps=[

        (
            "markdown_fill",
            MarkdownFiller()
        ),


        (
            "preprocessor",
            preprocessor
        ),


        (
            "model",
            xgb_model
        )

    ]
)

In [ ]:
from sklearn.metrics import make_scorer


def wmae(y_true, y_pred):
    weights = np.where(
        val_df["IsHoliday"],
        5,
        1
    )

    return np.sum(
        weights * np.abs(y_true - y_pred)
    ) / np.sum(weights)


wmae_scorer = make_scorer(
    wmae,
    greater_is_better=False
)

In [ ]:
param_grid = {

    "model__n_estimators": [
        700,
        1000,
        1300
    ],

    "model__learning_rate": [
        0.02,
        0.03,
        0.05
    ],

    "model__max_depth": [
        2,
        3
    ],

    "model__min_child_weight": [
        10,
        20,
        30
    ],

    "model__gamma": [
        2,
        3,
        5
    ],

    "model__subsample": [
        0.7,
        0.8
    ],

    "model__colsample_bytree": [
        0.6,
        0.7
    ],

    "model__reg_lambda": [
        5,
        10,
        20
    ]
}

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(
    n_splits=3
)

In [ ]:
random_search = RandomizedSearchCV(

    estimator=pipeline,

    param_distributions=param_grid,

    n_iter=50,

    scoring=wmae_scorer,

    cv=tscv,

    verbose=2,

    random_state=42,

    n_jobs=-1
)

In [ ]:
random_search.fit(
    train_df,
    y_train

)

Fitting 3 folds for each of 50 candidates, totalling 150 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


RandomizedSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=3, test_size=None),
                   estimator=Pipeline(steps=[('markdown_fill',
                                              MarkdownFiller()),
                                             ('preprocessor',
                                              ColumnTransformer(transformers=[('categorical',
                                                                               OneHotEncoder(handle_unknown='ignore'),
                                                                               <sklearn.compose._column_transformer.make_column_selector object at 0x7b0dbd6de480>),
                                                                              ('numerical',
                                                                               S...
                                        'model__gamma': [1, 2, 3],
                                        'model__learning_rate': [0.02, 0.03,
                                                                 0.05],
                                        'model__max_depth': [2, 3, 4],
                                        'model__min_child_weight': [10, 20, 30],
                                        'model__n_estimators': [700, 900, 1200],
                                        'model__reg_alpha': [0, 0.1, 0.5],
                                        'model__reg_lambda': [5, 10, 20],
                                        'model__subsample': [0.7, 0.8, 0.9]},
                   random_state=42,
                   scoring=make_scorer(wmae, greater_is_better=False, response_method='predict'),
                   verbose=2)

In [ ]:
print(
    random_search.best_params_
)

{'model__subsample': 0.7, 'model__reg_lambda': 5, 'model__reg_alpha': 0, 'model__n_estimators': 700, 'model__min_child_weight': 10, 'model__max_depth': 4, 'model__learning_rate': 0.02, 'model__gamma': 2, 'model__colsample_bytree': 0.8}


In [ ]:
best_model = random_search.best_estimator_

In [ ]:
train_pred = best_model.predict(train_df)
val_pred = best_model.predict(val_df)

In [ ]:
pipeline.fit(train_df, y_train)

Pipeline(steps=[('markdown_fill', MarkdownFiller()),
                ('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x783ec9740980>),
                                                 ('numerical',
                                                  SimpleImputer(strategy='median'),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x78...
                              gamma=0.6607231426966449, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None,
                              learning_rate=0.11667242986570267, max_bin=None,
                              max_cat_threshold=None, max_cat_to_onehot=None,
                              max_delta_step=None, max_depth=9, max_leaves=None,
                              min_child_weight=3, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=1671, n_jobs=None,
                              num_parallel_tree=None, ...))])

In [ ]:
train_pred = pipeline.predict(train_df)
val_pred = pipeline.predict(val_df)

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:553: UserWarning: [09:36:05] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [ ]:
train_mae = mean_absolute_error(
    y_train,
    train_pred
)

val_mae = mean_absolute_error(
    y_val,
    val_pred
)


train_weights = np.where(
    train_df["IsHoliday"],
    5,
    1
)

val_weights = np.where(
    val_df["IsHoliday"],
    5,
    1
)


train_wmae = (
    np.sum(train_weights * np.abs(y_train - train_pred))
    /
    np.sum(train_weights)
)


val_wmae = (
    np.sum(val_weights * np.abs(y_val - val_pred))
    /
    np.sum(val_weights)
)


print("Train MAE:", train_mae)
print("Train WMAE:", train_wmae)

print("Val MAE:", val_mae)
print("Val WMAE:", val_wmae)

Train MAE: 1248.1584580300978
Train WMAE: 1250.1610548710555
Val MAE: 1754.9358695273725
Val WMAE: 1995.0888451169924


In [ ]:
mlflow.set_experiment("XGBoost_Training")
with mlflow.start_run(
    run_name="XGBoost_Hyperparameter_Search_Run_12"
):

    mlflow.log_params(
        random_search.best_params_
    )

    mlflow.log_metrics({
        "train_MAE": train_mae,
        "train_WMAE": train_wmae,
        "val_MAE": val_mae,
        "val_WMAE": val_wmae
    })



🏃 View run XGBoost_Hyperparameter_Search_Run_12 at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2/runs/80a2a5717ffc454f9f809f93469c4c33
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/2


In [14]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin



train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
stores = pd.read_csv("stores.csv")
features = pd.read_csv("features.csv")

merged_train = (
    train
    .merge(stores, on="Store", how="left")
    .merge(features, on=["Store", "Date", "IsHoliday"], how="left")
)

merged_test = (
    test
    .merge(stores, on="Store", how="left")
    .merge(features, on=["Store", "Date", "IsHoliday"], how="left")
)

merged_train["IsTest"] = 0
merged_test["IsTest"] = 1
merged_test["Weekly_Sales"] = np.nan

combined = pd.concat([merged_train, merged_test], ignore_index=True)

combined["Date"] = pd.to_datetime(combined["Date"])
combined = combined.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
combined = combined.replace([np.inf, -np.inf], np.nan)



class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.min_date_ = pd.to_datetime(X["Date"]).min()
        return self

    def transform(self, X):
        X = X.copy()
        X["Date"] = pd.to_datetime(X["Date"])

        X["Lag_52"] = (
            X.groupby(["Store", "Dept"], observed=True)["Weekly_Sales"]
            .shift(52) if "Weekly_Sales" in X.columns else None
        )

        if "Lag_52" in X.columns:
            X["Holiday_Lag52"] = X["IsHoliday"].astype(int) * X["Lag_52"]
        else:
            X["Holiday_Lag52"] = 0

        X["Year"] = X["Date"].dt.year
        X["Month"] = X["Date"].dt.month
        X["Week"] = X["Date"].dt.isocalendar().week.astype(int)
        X["Quarter"] = X["Date"].dt.quarter
        X["Days_from_start"] = (X["Date"] - self.min_date_).dt.days

        return X


class TargetNormalizer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        temp_df = X.copy()
        temp_df["Weekly_Sales"] = y

        holiday_stats = temp_df.groupby(["Store", "Dept", "IsHoliday"])["Weekly_Sales"].mean().unstack(fill_value=0)
        non_holiday_mean = holiday_stats[0] if 0 in holiday_stats.columns else pd.Series(1.0, index=holiday_stats.index)
        holiday_mean = holiday_stats[1] if 1 in holiday_stats.columns else pd.Series(1.0, index=holiday_stats.index)

        lift = holiday_mean / np.maximum(non_holiday_mean, 1e-5)
        self.lift_dict_ = lift.replace([np.inf, np.nan], 1.4).clip(lower=1.1, upper=2.5).to_dict()
        return self

    def transform(self, X):
        return X.copy()


class MarkdownFiller(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.markdown_cols = [
            "MarkDown1",
            "MarkDown2",
            "MarkDown3",
            "MarkDown4",
            "MarkDown5"
        ]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.markdown_cols:
            if col in X.columns:
                X[col] = X[col].fillna(0)
        return X


class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns_to_drop):
        self.columns_to_drop = columns_to_drop

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        cols_present = [col for col in self.columns_to_drop if col in X.columns]
        return X.drop(columns=cols_present)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            make_column_selector(dtype_include=["object", "category", "bool"])
        ),
        (
            "numerical",
            SimpleImputer(strategy="median"),
            make_column_selector(dtype_include=np.number)
        )
    ]
)



fe = FeatureEngineer()
combined = fe.fit_transform(combined)

train_full = combined[combined["IsTest"] == 0].copy()
test_final = combined[combined["IsTest"] == 1].copy()

train_full = train_full.dropna(subset=["Lag_52"]).copy()

y_full = train_full.pop("Weekly_Sales")
X_full = train_full.drop(columns=["IsTest"])
X_test = test_final.drop(columns=["IsTest", "Weekly_Sales"])



normalizer = TargetNormalizer()
normalizer.fit(X_full, y_full)

full_lifts = list(zip(X_full["Store"], X_full["Dept"]))
full_lift_series = pd.Series(full_lifts).map(normalizer.lift_dict_).fillna(1.4).values

y_full_base = np.where(
    X_full["IsHoliday"] == 1,
    y_full / full_lift_series,
    y_full
)



mlflow.set_experiment("Walmart_XGB_Pipeline")

with mlflow.start_run(run_name="XGB_Full_Pipeline"):

    params = {
        "objective": "reg:squarederror",
        "random_state": 42,
        "tree_method": "hist",
        "device": "cuda",
        "subsample": 0.7,
        "reg_lambda": 5,
        "reg_alpha": 0,
        "n_estimators": 700,
        "min_child_weight": 10,
        "max_depth": 4,
        "learning_rate": 0.02,
        "gamma": 2,
        "colsample_bytree": 0.8
    }

    mlflow.log_params(params)

    pipeline = Pipeline(
        steps=[
            ("markdown_fill", MarkdownFiller()),
            ("drop_unnecessary_cols", DropColumns(["Date", "Weekly_Sales"])),
            ("preprocessor", preprocessor),
            ("model", XGBRegressor(**params))
        ]
    )

    pipeline.fit(X_full, y_full_base)

    mlflow.sklearn.log_model(
        pipeline,
        "xgb_pipeline_model",
        skops_trusted_types=[
            "__main__.DropColumns",
            "__main__.MarkdownFiller",
            "collections.OrderedDict",
            "xgboost.core.Booster",
            "xgboost.sklearn.XGBRegressor",
            "numpy.dtype",
            "numpy.number",
            "sklearn.compose._column_transformer.make_column_selector"
        ]
    )

    test_preds_base = pipeline.predict(X_test)

    test_lifts = list(zip(X_test["Store"], X_test["Dept"]))
    test_lift_series = pd.Series(test_lifts).map(normalizer.lift_dict_).fillna(1.4).values

    final_test_preds = np.where(
        X_test["IsHoliday"] == 1,
        test_preds_base * test_lift_series,
        test_preds_base
    )



submission = pd.DataFrame({
    "Store": X_test["Store"],
    "Dept": X_test["Dept"],
    "Date": X_test["Date"].dt.strftime("%Y-%m-%d"),
    "Weekly_Sales": final_test_preds
})

submission["Id"] = submission["Store"].astype(str) + "_" + submission["Dept"].astype(str) + "_" + submission["Date"].astype(str)
submission = submission[["Id", "Weekly_Sales"]]

submission.to_csv("submission.csv", index=False)

print("=" * 60)
print("SUCCESS: XGBoost Model safely saved to MLflow & submission.csv generated!")
print(submission.head(10))
print("=" * 60)

2026/07/25 14:07:13 INFO mlflow.tracking.fluent: Experiment with name 'Walmart_XGB_Pipeline' does not exist. Creating a new experiment.
2026/07/25 14:07:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/25 14:07:26 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpgmvicdv3/model/model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.6.1', 'skops==0.14.0']. Set logging level to DEBUG to see the full traceback. 


🏃 View run XGB_Full_Pipeline at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/25/runs/f99a189c8b6d429ca7355c6301117aa9
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/25
SUCCESS: XGBoost Model safely saved to MLflow & submission.csv generated!
                 Id  Weekly_Sales
143  1_1_2012-11-02  38516.921875
144  1_1_2012-11-09  19662.265625
145  1_1_2012-11-16  20899.582031
146  1_1_2012-11-23  22680.938672
147  1_1_2012-11-30  26966.449219
148  1_1_2012-12-07  34187.441406
149  1_1_2012-12-14  42540.707031
150  1_1_2012-12-21  44403.632812
151  1_1_2012-12-28  27213.357617
152  1_1_2013-01-04  13606.968750
